In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings("ignore")


# ============================================================================
# Model Definition (must match training architecture)
# ============================================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_output = gru_out[:, -1, :]
        output = self.fc(last_output)
        return output


class TransformerModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(TransformerModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        effective_hidden = hidden_size * 2
        self.input_projection = nn.Linear(input_size, effective_hidden)
        self.positional_encoding = nn.Parameter(
            torch.zeros(1, 5000, effective_hidden), requires_grad=True
        )
        nn.init.normal_(self.positional_encoding, mean=0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=effective_hidden,
            nhead=8,
            dim_feedforward=effective_hidden * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        effective_num_layers = max(2, num_layers)
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=effective_num_layers,
            norm=nn.LayerNorm(effective_hidden),
        )

        self.fc = nn.Sequential(
            nn.Linear(effective_hidden, effective_hidden // 2),
            nn.LayerNorm(effective_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(effective_hidden // 2, output_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        _, seq_len, _ = x.shape
        x = self.input_projection(x)
        x = x + self.positional_encoding[:, :seq_len, :]
        x = self.dropout(x)
        transformer_out = self.transformer_encoder(x)
        last_output = transformer_out[:, -1, :]
        output = self.fc(last_output)

        return output


# ============================================================================
# Dataset Class (same as training)
# ============================================================================
class SlidingWindowDataset(Dataset):
    """Sliding window dataset with gap handling and feature engineering"""

    def __init__(
        self,
        dataframe,
        window_minutes=60,
        forecast_minutes=30,
        data_interval_minutes=5,
        max_gap_minutes=10,
        scalers=None,
        fit_scalers=True,
        debug=False,
        verbose=True,
    ):
        self.df = dataframe.copy().sort_values("timestamp").reset_index(drop=True)
        self.window_minutes = window_minutes
        self.forecast_minutes = forecast_minutes
        self.data_interval = data_interval_minutes
        self.max_gap = max_gap_minutes
        self.fit_scalers = fit_scalers
        self.debug = debug
        self.verbose = verbose

        self.window_size = window_minutes // data_interval_minutes
        self.forecast_step = forecast_minutes // data_interval_minutes

        self._impute_missing_values()
        self._identify_gaps()
        self._extract_features()
        self._normalize_features(scalers)
        self.valid_indices = self._get_valid_indices()

        if self.verbose:
            self._validate_data_quality()

    def _impute_missing_values(self):
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"])
        if "glucose" not in self.df.columns:
            return

        is_na = self.df["glucose"].isna()
        if not is_na.any():
            if self.verbose:
                print("No missing values in glucose")
            return

        na_groups = (is_na != is_na.shift()).cumsum()
        self.df["_na_groups"] = na_groups

        filled_count = 0
        kept_count = 0

        for group_id in self.df[is_na]["_na_groups"].unique():
            group_mask = (self.df["_na_groups"] == group_id) & is_na
            group_indices = self.df[group_mask].index.tolist()

            if not group_indices:
                continue

            gap_duration = len(group_indices) * self.data_interval

            if gap_duration <= self.max_gap:
                self._fill_gap("glucose", group_indices, is_na)
                filled_count += len(group_indices)
            else:
                kept_count += len(group_indices)

        self.df.drop("_na_groups", axis=1, inplace=True, errors="ignore")

        if self.verbose:
            print(
                f"Glucose imputation: filled {filled_count} NaN, kept {kept_count} NaN for filtering"
            )

    def _fill_gap(self, feature, indices, is_na):
        start_idx = indices[0]
        end_idx = indices[-1]

        prev_valid_idx = start_idx - 1
        while prev_valid_idx >= 0 and is_na.iloc[prev_valid_idx]:
            prev_valid_idx -= 1

        next_valid_idx = end_idx + 1
        while next_valid_idx < len(self.df) and is_na.iloc[next_valid_idx]:
            next_valid_idx += 1

        if prev_valid_idx >= 0 and next_valid_idx < len(self.df):
            prev_val = self.df.iloc[prev_valid_idx][feature]
            next_val = self.df.iloc[next_valid_idx][feature]

            for idx in range(start_idx, end_idx + 1):
                ratio = (idx - prev_valid_idx) / (next_valid_idx - prev_valid_idx)
                self.df.at[self.df.index[idx], feature] = (
                    prev_val + (next_val - prev_val) * ratio
                )

        elif prev_valid_idx >= 0:
            self._extrapolate_forward(
                feature, start_idx, end_idx, prev_valid_idx, is_na
            )
        elif next_valid_idx < len(self.df):
            self._extrapolate_backward(
                feature, start_idx, end_idx, next_valid_idx, is_na
            )

    def _extrapolate_forward(self, feature, start_idx, end_idx, last_valid_idx, is_na):
        second_last_idx = last_valid_idx - 1
        while second_last_idx >= 0 and is_na.iloc[second_last_idx]:
            second_last_idx -= 1

        if second_last_idx >= 0:
            val2 = self.df.iloc[second_last_idx][feature]
            val1 = self.df.iloc[last_valid_idx][feature]
            slope = val1 - val2

            for idx in range(start_idx, end_idx + 1):
                steps = idx - last_valid_idx
                self.df.at[self.df.index[idx], feature] = val1 + slope * steps
        else:
            fill_val = self.df.iloc[last_valid_idx][feature]
            self.df.loc[self.df.index[start_idx : end_idx + 1], feature] = fill_val

    def _extrapolate_backward(
        self, feature, start_idx, end_idx, first_valid_idx, is_na
    ):
        second_valid_idx = first_valid_idx + 1
        while second_valid_idx < len(self.df) and is_na.iloc[second_valid_idx]:
            second_valid_idx += 1

        if second_valid_idx < len(self.df):
            val1 = self.df.iloc[first_valid_idx][feature]
            val2 = self.df.iloc[second_valid_idx][feature]
            slope = val2 - val1

            for idx in range(start_idx, end_idx + 1):
                steps = idx - first_valid_idx
                self.df.at[self.df.index[idx], feature] = val1 + slope * steps
        else:
            fill_val = self.df.iloc[first_valid_idx][feature]
            self.df.loc[self.df.index[start_idx : end_idx + 1], feature] = fill_val

    def _identify_gaps(self):
        self.df["timestamp"] = pd.to_datetime(self.df["timestamp"])
        self.df["time_diff"] = self.df["timestamp"].diff().dt.total_seconds() / 60
        self.df["is_gap"] = self.df["time_diff"] > self.max_gap
        self.df["gap_id"] = self.df["is_gap"].cumsum()

    def _extract_features(self):
        self.df["hour"] = self.df["timestamp"].dt.hour
        self.df["minute"] = self.df["timestamp"].dt.minute
        self.df["day_of_week"] = self.df["timestamp"].dt.dayofweek
        self.df["is_weekend"] = self.df["day_of_week"].isin([5, 6]).astype(float)

        self.df["hour_sin"] = np.sin(2 * np.pi * self.df["hour"] / 24)
        self.df["hour_cos"] = np.cos(2 * np.pi * self.df["hour"] / 24)

        self.numeric_features = ["glucose", "insulin", "meal_carbs"]
        self.time_features = ["hour_sin", "hour_cos", "is_weekend"]
        self.all_features = self.numeric_features + self.time_features

    def _get_valid_indices(self):
        valid_indices = []
        self.drop_stats = {
            "segment_too_short": 0,
            "window_has_nan": 0,
            "target_is_nan": 0,
        }

        for _, group_data in self.df.groupby("gap_id"):
            group_indices = group_data.index.tolist()

            if len(group_indices) < self.window_size + self.forecast_step:
                self.drop_stats["segment_too_short"] += len(group_indices)
                continue

            max_start_idx = len(group_indices) - self.window_size - self.forecast_step

            for i in range(max_start_idx + 1):
                start_idx = group_indices[i]
                window_end_idx = start_idx + self.window_size - 1
                target_idx = start_idx + self.window_size + self.forecast_step - 1

                window_glucose = self.df.loc[start_idx:window_end_idx, "glucose"]
                if window_glucose.isna().any():
                    self.drop_stats["window_has_nan"] += 1
                    continue

                target_value = self.df.loc[target_idx, "glucose"]
                if pd.isna(target_value):
                    self.drop_stats["target_is_nan"] += 1
                    continue

                valid_indices.append(start_idx)

        return valid_indices

    def _normalize_features(self, external_scalers=None):
        self.scalers = {}

        for feature in self.numeric_features:
            if self.fit_scalers:
                scaler = RobustScaler()
                non_na_mask = self.df[feature].notna()

                if non_na_mask.any():
                    self.df.loc[non_na_mask, feature] = scaler.fit_transform(
                        self.df.loc[non_na_mask, feature].values.reshape(-1, 1)
                    ).flatten()
                self.scalers[feature] = scaler
            else:
                if external_scalers is None or feature not in external_scalers:
                    raise ValueError(f"Missing scaler for: {feature}")

                scaler = external_scalers[feature]
                non_na_mask = self.df[feature].notna()

                if non_na_mask.any():
                    self.df.loc[non_na_mask, feature] = scaler.transform(
                        self.df.loc[non_na_mask, feature].values.reshape(-1, 1)
                    ).flatten()
                self.scalers[feature] = scaler

        self.feature_data = torch.tensor(
            self.df[self.all_features].values, dtype=torch.float32
        )
        self.target_data = torch.tensor(self.df["glucose"].values, dtype=torch.float32)

    def _validate_data_quality(self):
        print(f"\n{'=' * 60}")
        print("Data Quality Report")
        print(f"{'=' * 60}")

        print("\n【Original Missing Values】")
        for feature in ["glucose", "insulin", "meal_carbs"]:
            if feature in self.df.columns:
                nan_count = self.df[feature].isna().sum()
                nan_pct = 100 * nan_count / len(self.df) if len(self.df) > 0 else 0
                print(f"  {feature:12s}: {nan_count:5d} ({nan_pct:5.2f}%)")

        print("\n【Sample Generation】")
        print(f"  Total rows:       {len(self.df):5d}")
        print(f"  Valid samples:    {len(self.valid_indices):5d}")

        if len(self.df) > 0:
            drop_rate = 100 * (1 - len(self.valid_indices) / len(self.df))
            print(f"  Drop rate:        {drop_rate:5.2f}%")

        print(f"{'=' * 60}\n")

    def get_scalers(self):
        return self.scalers

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        x = self.feature_data[actual_idx : actual_idx + self.window_size]
        y = self.target_data[actual_idx + self.window_size + self.forecast_step - 1]

        if self.debug and (torch.isnan(x).any() or torch.isnan(y)):
            raise ValueError(f"NaN detected in sample {idx}")

        return x, y


# ============================================================================
# Prediction and Visualization Functions
# ============================================================================
class GlucosePredictor:
    """Load trained model and generate predictions with visualizations"""

    def __init__(self, model_path, device=None):
        self.device = (
            device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.model_path = model_path
        self.model = None
        self.scalers = None
        self.dataset_config = None
        self.model_config = None
        self.model_type = None  # 新增:存储模型类型

        self._load_model()

    def _load_model(self):
        """Load saved model and configurations"""
        print("\n" + "=" * 80)
        print(f"Loading model from: {self.model_path}")
        print("=" * 80)

        checkpoint = torch.load(
            self.model_path, map_location=self.device, weights_only=False
        )

        # Extract configurations
        self.model_config = checkpoint["model_config"]
        self.scalers = checkpoint["scalers"]
        self.dataset_config = checkpoint["dataset_config"]

        # 根据文件名确定模型类型
        if "LSTM" in self.model_path:
            self.model_type = "LSTM"
            self.model = LSTMModel(
                input_size=self.model_config["input_size"],
                hidden_size=self.model_config["hidden_size"],
                num_layers=self.model_config["num_layers"],
                output_size=1,
                dropout=0.0,
            ).to(self.device)
        elif "GRU" in self.model_path:
            self.model_type = "GRU"
            self.model = GRUModel(
                input_size=self.model_config["input_size"],
                hidden_size=self.model_config["hidden_size"],
                num_layers=self.model_config["num_layers"],
                output_size=1,
                dropout=0.0,
            ).to(self.device)
        elif "Transformer" in self.model_path:
            self.model_type = "Transformer"
            self.model = TransformerModel(
                input_size=self.model_config["input_size"],
                hidden_size=self.model_config["hidden_size"],
                num_layers=self.model_config["num_layers"],
                output_size=1,
                dropout=0.0,
            ).to(self.device)
        else:
            raise ValueError(f"Unknown model type in path: {self.model_path}")

        # Load weights
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.eval()

        print("✓ Model loaded successfully")
        print(
            f"  - Architecture: {self.model_type} with {self.model_config['num_layers']} layers"
        )
        print(f"  - Hidden size: {self.model_config['hidden_size']}")
        print(f"  - Input size: {self.model_config['input_size']}")
        print(f"  - Window: {self.dataset_config['window_minutes']} minutes")
        print(
            f"  - Forecast horizon: {self.dataset_config['forecast_minutes']} minutes"
        )
        print("=" * 80)

    def predict_patient(self, patient_df, patient_id, batch_size=64):
        """Generate predictions for a single patient"""
        print(f"\n{'─' * 60}")
        print(f"Predicting for Patient {patient_id}")
        print(f"{'─' * 60}")

        # Create dataset
        test_dataset = SlidingWindowDataset(
            dataframe=patient_df,
            window_minutes=self.dataset_config["window_minutes"],
            forecast_minutes=self.dataset_config["forecast_minutes"],
            data_interval_minutes=self.dataset_config["data_interval"],
            max_gap_minutes=self.dataset_config["max_gap"],
            scalers=self.scalers,
            fit_scalers=False,
            verbose=True,
        )

        if len(test_dataset) == 0:
            print(f"⚠ No valid samples for patient {patient_id}")
            return None

        # Create dataloader
        test_loader = DataLoader(
            test_dataset, batch_size=batch_size, shuffle=False, num_workers=0
        )

        # Generate predictions
        predictions_scaled = []
        targets_scaled = []

        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_batch = x_batch.to(self.device)
                pred = self.model(x_batch).squeeze()
                predictions_scaled.extend(pred.cpu().numpy())
                targets_scaled.extend(y_batch.numpy())

        # Convert to arrays
        predictions_scaled = np.array(predictions_scaled).reshape(-1, 1)
        targets_scaled = np.array(targets_scaled).reshape(-1, 1)

        # Inverse transform to original scale
        glucose_scaler = self.scalers["glucose"]
        predictions = glucose_scaler.inverse_transform(predictions_scaled).flatten()
        targets = glucose_scaler.inverse_transform(targets_scaled).flatten()

        # Calculate metrics
        mae = np.mean(np.abs(targets - predictions))
        rmse = np.sqrt(np.mean((targets - predictions) ** 2))
        mape = np.mean(np.abs((targets - predictions) / targets)) * 100
        r2 = 1 - (
            np.sum((targets - predictions) ** 2)
            / np.sum((targets - np.mean(targets)) ** 2)
        )

        print("\n📊 Performance Metrics:")
        print(f"  - MAE:  {mae:.2f} mg/dL")
        print(f"  - RMSE: {rmse:.2f} mg/dL")
        print(f"  - MAPE: {mape:.2f}%")
        print(f"  - R²:   {r2:.4f}")
        print(f"  - Samples: {len(predictions)}")

        return {
            "predictions": predictions,
            "targets": targets,
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
            "r2": r2,
            "n_samples": len(predictions),
        }

    def plot_predictions(self, results_dict, save_dir="plots", show_plot=True):
        """Create comprehensive visualization of predictions vs actual values"""
        import os

        os.makedirs(save_dir, exist_ok=True)

        num_patients = len(results_dict)
        fig, axes = plt.subplots(
            num_patients,
            2,
            figsize=(18, 4 * num_patients),
            gridspec_kw={"width_ratios": [3, 1]},
        )

        # Handle single patient case
        if num_patients == 1:
            axes = axes.reshape(1, -1)

        for idx, (patient_id, results) in enumerate(results_dict.items()):
            if results is None:
                continue

            predictions = results["predictions"]
            targets = results["targets"]

            # Left subplot: Time series
            ax1 = axes[idx, 0]
            time_indices = np.arange(len(predictions))

            ax1.plot(
                time_indices,
                targets,
                label="Actual Glucose",
                color="#2E86AB",
                linewidth=2,
                alpha=0.8,
            )
            ax1.plot(
                time_indices,
                predictions,
                label="Predicted Glucose",
                color="#A23B72",
                linewidth=2,
                alpha=0.8,
                linestyle="--",
            )

            # Add clinical zones
            ax1.axhspan(
                70, 180, alpha=0.1, color="green", label="Target Range (70-180)"
            )
            ax1.axhspan(0, 70, alpha=0.1, color="red", label="Hypoglycemia (<70)")
            ax1.axhspan(
                180, 400, alpha=0.1, color="orange", label="Hyperglycemia (>180)"
            )

            ax1.set_xlabel("Sample Index", fontsize=11, fontweight="bold")
            ax1.set_ylabel("Glucose (mg/dL)", fontsize=11, fontweight="bold")
            ax1.set_title(
                f"Patient {patient_id} - Glucose Prediction (30-min Horizon)\n"
                f"MAE: {results['mae']:.2f} mg/dL | RMSE: {results['rmse']:.2f} mg/dL | R²: {results['r2']:.4f}",
                fontsize=12,
                fontweight="bold",
            )
            ax1.legend(loc="upper right", fontsize=9)
            ax1.grid(True, alpha=0.3, linestyle="--")
            ax1.set_ylim(0, 400)

            # Right subplot: Scatter plot
            ax2 = axes[idx, 1]
            ax2.scatter(
                targets,
                predictions,
                alpha=0.5,
                s=20,
                color="#F18F01",
                edgecolors="black",
                linewidths=0.5,
            )

            # Perfect prediction line
            min_val = min(targets.min(), predictions.min())
            max_val = max(targets.max(), predictions.max())
            ax2.plot(
                [min_val, max_val],
                [min_val, max_val],
                "r--",
                linewidth=2,
                label="Perfect Prediction",
            )

            ax2.set_xlabel("Actual Glucose (mg/dL)", fontsize=11, fontweight="bold")
            ax2.set_ylabel("Predicted Glucose (mg/dL)", fontsize=11, fontweight="bold")
            ax2.set_title("Prediction vs Actual", fontsize=12, fontweight="bold")
            ax2.legend(loc="upper left", fontsize=9)
            ax2.grid(True, alpha=0.3, linestyle="--")
            ax2.set_aspect("equal", adjustable="box")

        plt.tight_layout()

        # Save figure
        save_path = os.path.join(save_dir, "glucose_predictions_all_patients.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"\n✓ Plot saved to: {save_path}")

        if show_plot:
            plt.show()
        else:
            plt.close()

    def plot_individual_patient(
        self, patient_id, results, save_dir="plots", show_plot=True, max_samples=600
    ):
        """Create detailed plot for a single patient with zoomed view"""
        import os

        os.makedirs(save_dir, exist_ok=True)

        if results is None:
            print(f"⚠ No results available for patient {patient_id}")
            return

        predictions = results["predictions"]
        targets = results["targets"]

        # Limit samples for better visualization
        if len(predictions) > max_samples:
            predictions = predictions[:max_samples]
            targets = targets[:max_samples]

        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

        # Main time series plot
        ax1 = fig.add_subplot(gs[0:2, :])
        time_indices = np.arange(len(predictions))

        ax1.plot(
            time_indices,
            targets,
            label="Actual Glucose",
            color="#2E86AB",
            linewidth=2.5,
            marker="o",
            markersize=3,
            alpha=0.8,
        )
        ax1.plot(
            time_indices,
            predictions,
            label="Predicted Glucose",
            color="#A23B72",
            linewidth=2.5,
            marker="s",
            markersize=3,
            alpha=0.8,
            linestyle="--",
        )

        # Clinical zones
        ax1.axhspan(70, 180, alpha=0.1, color="green", label="Target Range")
        ax1.axhspan(0, 70, alpha=0.1, color="red")
        ax1.axhspan(180, 400, alpha=0.1, color="orange")

        ax1.set_xlabel("Sample Index", fontsize=13, fontweight="bold")
        ax1.set_ylabel("Glucose (mg/dL)", fontsize=13, fontweight="bold")
        ax1.set_title(
            f"Patient {patient_id} - 30-Minute Glucose Prediction\n"
            f"MAE: {results['mae']:.2f} mg/dL | RMSE: {results['rmse']:.2f} mg/dL | "
            f"R²: {results['r2']:.4f}",
            fontsize=15,
            fontweight="bold",
            pad=20,
        )
        ax1.legend(loc="upper right", fontsize=11, framealpha=0.9)
        ax1.grid(True, alpha=0.3, linestyle="--")
        ax1.set_ylim(0, 400)

        # Scatter plot
        ax2 = fig.add_subplot(gs[2, 0])
        ax2.scatter(
            targets,
            predictions,
            alpha=0.6,
            s=30,
            color="#F18F01",
            edgecolors="black",
            linewidths=0.5,
        )

        min_val = min(targets.min(), predictions.min())
        max_val = max(targets.max(), predictions.max())
        ax2.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Perfect Prediction",
            alpha=0.8,
        )

        ax2.set_xlabel("Actual Glucose (mg/dL)", fontsize=11, fontweight="bold")
        ax2.set_ylabel("Predicted Glucose (mg/dL)", fontsize=11, fontweight="bold")
        ax2.set_title("Prediction vs Actual", fontsize=12, fontweight="bold")
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        ax2.set_aspect("equal", adjustable="box")

        # Error distribution
        ax3 = fig.add_subplot(gs[2, 1])
        errors = predictions - targets
        ax3.hist(errors, bins=50, color="#6A4C93", alpha=0.7, edgecolor="black")
        ax3.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero Error")
        ax3.axvline(
            np.mean(errors),
            color="green",
            linestyle="--",
            linewidth=2,
            label=f"Mean Error: {np.mean(errors):.2f}",
        )

        ax3.set_xlabel("Prediction Error (mg/dL)", fontsize=11, fontweight="bold")
        ax3.set_ylabel("Frequency", fontsize=11, fontweight="bold")
        ax3.set_title("Error Distribution", fontsize=12, fontweight="bold")
        ax3.legend(fontsize=9)
        ax3.grid(True, alpha=0.3, axis="y")

        # Save figure
        save_path = os.path.join(save_dir, f"patient_{patient_id}_detailed.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"✓ Detailed plot saved to: {save_path}")

        if show_plot:
            plt.show()
        else:
            plt.close()


# ============================================================================
# Main Execution
# ============================================================================
def main():
    """Main execution function"""
    print("\n" + "=" * 80)
    print("BLOOD GLUCOSE PREDICTION - MODEL INFERENCE & VISUALIZATION")
    print("=" * 80)

    # Configuration
    MODEL_PATHS = [
        "save_model/LSTM_model_30_optimized.pth",
        "save_model/GRU_model_30_optimized.pth",
        "save_model/Transformer_model_30_optimized.pth",
    ]

    TEST_DATA_PATH = "ohio_t1dm_test_data.csv"
    PATIENT_IDS = [559, 563, 570, 575, 588, 591]
    FEATURE_COLS = ["glucose", "insulin", "meal_carbs", "timestamp"]
    BATCH_SIZE = 64
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    print("\n⚙ Configuration:")
    print(f"  - Device: {DEVICE}")
    print(f"  - Models: {len(MODEL_PATHS)}")
    for mp in MODEL_PATHS:
        print(f"    • {mp}")
    print(f"  - Test data: {TEST_DATA_PATH}")
    print(f"  - Patients: {PATIENT_IDS}")

    # Load test data (只需加载一次)
    print(f"\n{'─' * 80}")
    print("Loading test data...")
    test_df = pd.read_csv(TEST_DATA_PATH)
    print(f"✓ Loaded {len(test_df)} records")

    # Prepare patient-specific dataframes
    test_dfs = {}
    for pid in PATIENT_IDS:
        test_dfs[pid] = (
            test_df[test_df["patient_id"] == pid][FEATURE_COLS]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        print(f"  - Patient {pid}: {len(test_dfs[pid])} records")

    # 循环处理每个模型
    for model_path in MODEL_PATHS:
        # 根据模型文件名确定输出文件夹
        if "LSTM" in model_path:
            output_dir = "plots_LSTM"
            model_name = "LSTM"
        elif "GRU" in model_path:
            output_dir = "plots_GRU"
            model_name = "GRU"
        elif "Transformer" in model_path:
            output_dir = "plots_Transformer"
            model_name = "Transformer"
        else:
            continue

        print(f"\n{'=' * 80}")
        print(f"Processing {model_name} Model")
        print(f"{'=' * 80}")

        # Initialize predictor
        predictor = GlucosePredictor(model_path, device=DEVICE)

        # Generate predictions for all patients
        print(f"\n{'─' * 80}")
        print(f"Generating predictions for all patients using {model_name}...")
        results_dict = {}

        for pid in PATIENT_IDS:
            results = predictor.predict_patient(
                patient_df=test_dfs[pid], patient_id=pid, batch_size=BATCH_SIZE
            )
            results_dict[pid] = results

        # Create visualizations
        print(f"\n{'─' * 80}")
        print(f"Creating visualizations for {model_name}...")

        # Combined plot for all patients
        predictor.plot_predictions(
            results_dict=results_dict, save_dir=output_dir, show_plot=False
        )

        # Individual detailed plots
        for pid in PATIENT_IDS:
            if results_dict[pid] is not None:
                predictor.plot_individual_patient(
                    patient_id=pid,
                    results=results_dict[pid],
                    save_dir=output_dir,
                    show_plot=False,
                    max_samples=600,
                )

        # Summary statistics for current model
        print(f"\n{'=' * 80}")
        print(f"{model_name} MODEL SUMMARY")
        print(f"{'=' * 80}")

        print(
            f"\n{'Patient ID':^12} {'Samples':^10} {'MAE':^12} {'RMSE':^12} {'MAPE':^12} {'R²':^12}"
        )
        print("─" * 80)

        valid_results = {
            pid: res for pid, res in results_dict.items() if res is not None
        }

        all_mae = []
        all_rmse = []
        all_mape = []
        all_r2 = []

        for pid in PATIENT_IDS:
            if pid in valid_results:
                res = valid_results[pid]
                print(
                    f"{pid:^12} {res['n_samples']:^10} {res['mae']:^12.2f} {res['rmse']:^12.2f} {res['mape']:^12.2f} {res['r2']:^12.4f}"
                )

                all_mae.append(res["mae"])
                all_rmse.append(res["rmse"])
                all_mape.append(res["mape"])
                all_r2.append(res["r2"])

        print("─" * 80)

        # Overall statistics
        if len(all_mae) > 0:
            print(f"\n{'Overall Performance':^80}")
            print(f"  MAE:  {np.mean(all_mae):7.2f} ± {np.std(all_mae):7.2f} mg/dL")
            print(f"  RMSE: {np.mean(all_rmse):7.2f} ± {np.std(all_rmse):7.2f} mg/dL")
            print(f"  MAPE: {np.mean(all_mape):7.2f} ± {np.std(all_mape):7.2f}%")
            print(f"  R²:   {np.mean(all_r2):7.4f} ± {np.std(all_r2):7.4f}")
            print(f"\n  Total patients evaluated: {len(valid_results)}")
            print(
                f"  Total predictions: {sum(res['n_samples'] for res in valid_results.values())}"
            )

        print(f"\n{'=' * 80}")
        print(f"✓ {model_name} MODEL PROCESSING COMPLETED!")
        print(f"{'=' * 80}")
        print(f"\n📁 Outputs saved to: {output_dir}/")
        print("  - glucose_predictions_all_patients.png (comprehensive overview)")
        print("  - patient_[ID]_detailed.png (individual patient analysis)")

    # Final summary
    print(f"\n{'=' * 80}")
    print("✓ ALL MODELS PROCESSED SUCCESSFULLY!")
    print(f"{'=' * 80}")
    print("\n📁 Output directories:")
    print("  - plots_LSTM/")
    print("  - plots_GRU/")
    print("  - plots_Transformer/")
    print(f"\n{'=' * 80}\n")


if __name__ == "__main__":
    main()